# 08.1 — Classification Metrics

**Why accuracy is almost never the right metric:**
- 99% accuracy on fraud detection = just predict "not fraud" always (if 1% of transactions are fraud)
- Class imbalance is the norm, not the exception

**Build from scratch:** Confusion matrix → Precision → Recall → F1 → Macro/Micro/Weighted → ROC-AUC → PR curve

---

In [ ]:
import numpy as np
from typing import List, Optional
from collections import Counter

# ---- Confusion Matrix ----

def confusion_matrix(y_true: List[int], y_pred: List[int], 
                     n_classes: Optional[int] = None) -> np.ndarray:
    """
    Confusion matrix CM[i][j] = count of samples with true label i, predicted j.
    Rows = true labels, Columns = predicted labels.
    """
    if n_classes is None:
        n_classes = max(max(y_true), max(y_pred)) + 1
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm


def print_confusion_matrix(cm: np.ndarray, class_names: List[str] = None):
    n = cm.shape[0]
    if class_names is None:
        class_names = [str(i) for i in range(n)]
    col_width = max(max(len(c) for c in class_names), 6)
    header = " " * col_width + " | " + " | ".join(f"{c:>{col_width}}" for c in class_names)
    print("True\\Pred")
    print(header)
    print("-" * len(header))
    for i, row in enumerate(cm):
        print(f"{class_names[i]:>{col_width}} | " + " | ".join(f"{v:>{col_width}}" for v in row))


# Binary example: spam detection
y_true = [0,0,1,1,0,1,0,0,1,1,0,0,1,0,1]  # 0=ham, 1=spam
y_pred = [0,0,1,0,0,1,0,1,1,1,0,0,1,0,0]  # model predictions

cm = confusion_matrix(y_true, y_pred, n_classes=2)
print("Spam detection confusion matrix:")
print_confusion_matrix(cm, ["Ham", "Spam"])
print()
print(f"TP (Spam correctly detected) : {cm[1][1]}")
print(f"FP (Ham misclassified as Spam): {cm[0][1]}")
print(f"FN (Spam missed)              : {cm[1][0]}")
print(f"TN (Ham correctly ignored)    : {cm[0][0]}")

In [ ]:
# ---- Precision, Recall, F1 ----

def precision_recall_f1_per_class(cm: np.ndarray):
    """
    Per-class precision, recall, F1 from confusion matrix.
    
    Precision_i = TP_i / (TP_i + FP_i) = cm[i,i] / sum(cm[:,i])  (predicted positive)
    Recall_i    = TP_i / (TP_i + FN_i) = cm[i,i] / sum(cm[i,:])  (actually positive)
    F1_i        = 2 * P * R / (P + R)
    """
    n = cm.shape[0]
    results = {}
    for i in range(n):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp  # predicted class i but wrong
        fn = cm[i, :].sum() - tp  # actual class i but missed
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        support = cm[i, :].sum()
        results[i] = {'precision': p, 'recall': r, 'f1': f1, 'support': support}
    return results


def macro_f1(per_class: dict):
    """Unweighted average over classes. Treats all classes equally."""
    return np.mean([v['f1'] for v in per_class.values()])


def micro_f1(per_class: dict):
    """
    Aggregate TP, FP, FN across all classes, then compute F1.
    Equivalent to accuracy for multi-class with equal support.
    Dominated by frequent classes.
    """
    total_tp = sum(v['precision'] * v['support'] for v in per_class.values())
    # Actually compute from cm directly
    return 0.0  # placeholder — see below


def weighted_f1(per_class: dict):
    """Average weighted by class support. Good for imbalanced datasets."""
    total = sum(v['support'] for v in per_class.values())
    return sum(v['f1'] * v['support'] / total for v in per_class.values())


# Multi-class example: sentiment (Negative=0, Neutral=1, Positive=2)
y_true_mc = [0,1,2,0,2,1,0,2,2,1,0,1,2,0,2,1,0,2,1,0]
y_pred_mc = [0,1,2,1,2,1,0,1,2,0,0,2,2,0,1,1,0,2,2,0]

cm_mc = confusion_matrix(y_true_mc, y_pred_mc, n_classes=3)
print_confusion_matrix(cm_mc, ["Neg", "Neu", "Pos"])
print()

per_class = precision_recall_f1_per_class(cm_mc)
class_names = {0: "Negative", 1: "Neutral", 2: "Positive"}

print(f"{'Class':10} {'Precision':10} {'Recall':10} {'F1':10} {'Support':10}")
print("-" * 50)
for cls, m in per_class.items():
    print(f"{class_names[cls]:10} {m['precision']:.4f}     {m['recall']:.4f}     {m['f1']:.4f}     {m['support']}")

print()
print(f"Macro F1:    {macro_f1(per_class):.4f}  (treats all classes equally)")
print(f"Weighted F1: {weighted_f1(per_class):.4f}  (weights by support)")
print(f"Accuracy:    {sum(a==b for a,b in zip(y_true_mc,y_pred_mc))/len(y_true_mc):.4f}")

In [ ]:
# ---- ROC curve and AUC ----
# For binary classifiers that output probabilities.
# ROC = Receiver Operating Characteristic
# AUC = Area Under the Curve

def roc_curve_from_scratch(y_true: List[int], y_scores: List[float]):
    """
    Compute ROC curve.
    
    At each threshold t:
    - FPR = FP / (FP + TN) = false alarm rate
    - TPR = TP / (TP + FN) = recall
    
    Returns (fprs, tprs, thresholds)
    """
    # Sort by score descending — higher score = more confident positive
    sorted_pairs = sorted(zip(y_scores, y_true), reverse=True)
    
    total_pos = sum(y_true)
    total_neg = len(y_true) - total_pos
    
    fprs, tprs, thresholds = [0.0], [0.0], [sorted_pairs[0][0] + 1]
    tp, fp = 0, 0
    
    for score, label in sorted_pairs:
        if label == 1: tp += 1
        else: fp += 1
        fprs.append(fp / total_neg if total_neg > 0 else 0)
        tprs.append(tp / total_pos if total_pos > 0 else 0)
        thresholds.append(score)
    
    return np.array(fprs), np.array(tprs), np.array(thresholds)


def auc_trapezoid(fprs: np.ndarray, tprs: np.ndarray) -> float:
    """Area under ROC curve using trapezoidal rule."""
    return float(np.trapz(tprs, fprs))


# Generate synthetic binary classification scores
np.random.seed(42)
n = 200
y_true_bin = (np.random.rand(n) > 0.6).astype(int)  # ~40% positive
# Good classifier: positive class gets higher scores
y_scores_good = np.random.beta(4, 2, n) * y_true_bin + np.random.beta(2, 4, n) * (1 - y_true_bin)
# Random classifier
y_scores_rand = np.random.rand(n)

fprs_good, tprs_good, _ = roc_curve_from_scratch(y_true_bin.tolist(), y_scores_good.tolist())
fprs_rand, tprs_rand, _ = roc_curve_from_scratch(y_true_bin.tolist(), y_scores_rand.tolist())

auc_good = auc_trapezoid(fprs_good, tprs_good)
auc_rand = auc_trapezoid(fprs_rand, tprs_rand)

print(f"Good classifier AUC: {auc_good:.4f}  (closer to 1.0 = better)")
print(f"Random classifier AUC: {auc_rand:.4f}  (should be ~0.5)")
print()
print("AUC interpretation:")
print("  AUC = probability that a randomly chosen positive ranks higher than a random negative")
print("  AUC = 0.5: random. AUC = 1.0: perfect. AUC < 0.5: reversed (use 1-model)")

In [ ]:
# ---- Precision-Recall curve (better for imbalanced data) ----

def pr_curve(y_true: List[int], y_scores: List[float]):
    """
    Precision-Recall curve.
    More informative than ROC when positive class is rare.
    (ROC looks good even for bad models on imbalanced data — FPR hides TN)
    """
    sorted_pairs = sorted(zip(y_scores, y_true), reverse=True)
    total_pos = sum(y_true)
    
    precisions, recalls, thresholds = [], [], []
    tp, fp = 0, 0
    
    for score, label in sorted_pairs:
        if label == 1: tp += 1
        else: fp += 1
        precisions.append(tp / (tp + fp))
        recalls.append(tp / total_pos if total_pos > 0 else 0)
        thresholds.append(score)
    
    return np.array(precisions), np.array(recalls)


def average_precision(precisions: np.ndarray, recalls: np.ndarray) -> float:
    """Area under PR curve (AP = average precision)."""
    # Interpolated AP: sum of precision at each recall step
    ap = 0.0
    prev_recall = 0.0
    for p, r in zip(precisions, recalls):
        ap += p * (r - prev_recall)
        prev_recall = r
    return ap


precs_good, recs_good = pr_curve(y_true_bin.tolist(), y_scores_good.tolist())
precs_rand, recs_rand = pr_curve(y_true_bin.tolist(), y_scores_rand.tolist())

ap_good = average_precision(precs_good, recs_good)
ap_rand = average_precision(precs_rand, recs_rand)
baseline_ap = y_true_bin.mean()  # AP of random = prevalence

print(f"Good classifier AP:   {ap_good:.4f}")
print(f"Random classifier AP: {ap_rand:.4f}  (≈ baseline = {baseline_ap:.4f} = prevalence)")
print()
print("Precision@k: How many of the top-k predictions are correct?")
print("  Useful for recommendation systems, search results.")
print()
print("Choosing the operating threshold:")
print("  High precision / low recall:   flag only confident positives")
print("  High recall / low precision:   catch as many as possible (cancer screening)")
print("  Balance:                        F1 threshold = maximize F1 on val set")

# Find F1-optimal threshold
sorted_pairs = sorted(zip(y_scores_good, y_true_bin), reverse=True)
best_f1, best_threshold = 0, 0.5
tp = fp = 0
total_pos = y_true_bin.sum()
for score, label in sorted_pairs:
    if label == 1: tp += 1
    else: fp += 1
    fn = total_pos - tp
    p = tp / (tp + fp)
    r = tp / total_pos
    f1 = 2*p*r/(p+r) if p+r > 0 else 0
    if f1 > best_f1:
        best_f1, best_threshold = f1, score

print(f"\nF1-optimal threshold: {best_threshold:.3f}  (F1={best_f1:.4f})")

## Summary

| Metric | When to use |
|--------|-------------|
| Accuracy | Balanced classes, simple reporting |
| Macro F1 | Care about each class equally (even rare ones) |
| Weighted F1 | Care about overall performance, OK if rare classes matter less |
| ROC-AUC | Compare classifiers, threshold-independent, balanced classes |
| PR-AUC / AP | Imbalanced data, rare positive class |
| Precision@k | Recommendation, ranking, search |

**The metric IS the loss for your business problem.** Always ask: what does a wrong prediction cost? False positives vs false negatives are rarely symmetric.

---
**Next:** 08.2 — Generation Metrics (BLEU, ROUGE, BERTScore)